# CLEAR: SAM 3 for aerial scenes — Colab demo

1. 런타임 → 런타임 유형 변경 → GPU (L4 이상 권장)
2. Hugging Face에서 `facebook/sam3` 접근 신청 후, 왼쪽 열쇠(Secrets)에 `HF_TOKEN` 등록
3. 저장소가 private이면 GitHub 토큰을 `GH_TOKEN`으로 등록 (public이면 필요 없음)

In [ ]:
# [1] 환경 세팅
import os, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive, userdata
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/SAM3_PGLC'      # split과 점수 캐시가 저장되는 곳
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if not os.path.exists('/content/sam3'):
    !git clone -q https://github.com/facebookresearch/sam3 /content/sam3
!pip install -q --no-deps -e /content/sam3
!pip install -q timm ftfy iopath scikit-learn
sys.path.insert(0, '/content/sam3')

if not os.path.exists('/content/CLEAR_SAM3'):
    try:
        tok = userdata.get('GH_TOKEN')
        url = f'https://{tok}@github.com/ing0333/CLEAR_SAM3.git'
    except Exception:
        url = 'https://github.com/ing0333/CLEAR_SAM3.git'
    os.system(f'git clone -q {url} /content/CLEAR_SAM3')
sys.path.insert(0, '/content/CLEAR_SAM3')
print('setup done')

In [ ]:
# [2] 전체 재현: 점수 계산(GPU, 약 3시간, 끊겨도 이어서) → 표 계산(CPU, 수 초)
%cd /content/CLEAR_SAM3
!python experiments/run_pipeline.py --work_dir {WORK} --data_dir /content/aid --phrases prompts/phrases.json --out results/results.json

In [ ]:
# [3] 사진 한 장으로 OCT 효과 보기: 원래 이름 vs 바꾼 표현의 presence
import json
from sam3_rs.scoring import SAM3Scorer
from sam3_rs.data import load_img
split = json.load(open(f'{WORK}/split_seed0.json'))
f = next(f for f, c in split['val'] if c == 'Port')
phrases = json.load(open('prompts/phrases.json'))['phrases']['Port']
scorer = SAM3Scorer(ckpt_cache_dir='/content/hf_local')
pres = scorer.presence(load_img('/content/aid', f), scorer.encode_texts(phrases))
for p, s in zip(phrases, pres):
    print(f'{p:<22s} presence {s:.3f}')